In [ ]:
!pip install onnxruntime

In [ ]:
import glob
paths = glob.glob("/kaggle/input/**/*model.onnx", recursive=True)
print(paths)

In [ ]:
# =========================================================
# OFFICIAL EVALUATION NOTEBOOK — Imports
# =========================================================

import json
import os
import time
import glob
import numpy as np
import pandas as pd
from PIL import Image
import onnxruntime as ort   # ONNX inference engine

# =========================================================
# Required Paths
# =========================================================

# Path to evaluation images (provided by organizers)
IMAGE_DIR = "/kaggle/input/competitions/1st-krones-vision-ai-challenge/test_images"

# Path to your submitted ONNX model (inside your Kaggle dataset)
MODEL_PATH = "/kaggle/input/datasets/jonathanyung310/krones-efficientnet-b3/model.onnx"

# You may change this depending on your model requirements
IMG_SIZE = (512, 512)

# path to the test ROI annotation file
TEST_JSON = "/kaggle/input/competitions/1st-krones-vision-ai-challenge/test_annotations_roi_only.json"


print("Evaluation image directory exists:", os.path.exists(IMAGE_DIR))
print("ONNX model file exists:", os.path.exists(MODEL_PATH))
print("✅ Evaluation setup cell ran successfully")

def load_roi_map(annotation_file):
    with open(annotation_file, "r") as f:
        data = json.load(f)
    img_lookup = {img["id"]: img["file_name"] for img in data["images"]}
    roi_map = {}
    for ann in data["annotations"]:
        if ann.get("category_id") == 22:
            fname = img_lookup[ann["image_id"]]
            roi_map[fname] = ann["bbox"]
    print(f"✅ Loaded {len(roi_map)} ROI regions.")
    return roi_map

roi_map = load_roi_map(TEST_JSON)  # load the ROI map

In [ ]:
print("🔄 Loading ONNX model...")

# Create ONNX Runtime session
# GPU is attempted first; if unavailable, fallback to CPU occurs automatically.
session = ort.InferenceSession(
    MODEL_PATH,
    providers=[
        "CUDAExecutionProvider",  # Fastest execution if GPU is available
        "CPUExecutionProvider"    # Fallback (always supported)
    ]
)

# Extract the model's input name
input_name = session.get_inputs()[0].name

print(f"✅ ONNX model loaded successfully! | Input name: {input_name}")
print("ℹ️ Using providers:", session.get_providers())

In [ ]:
def preprocess_image(path):
    """
    Loads and preprocesses a single image.
    Modify this to match your model's expected input format.
    """

    # Load image and convert to RGB
    img = Image.open(path).convert("RGB")

    # the model trained on cropped images
    fname = os.path.basename(path)
    if fname in roi_map:
        x, y, w, h = roi_map[fname]
        img = img.crop((x, y, x + w, y + h))

    # Resize (change if your model uses letterboxing or different size)
    img = img.resize(IMG_SIZE)

    # Convert to NumPy array and normalize to [0,1]
    arr = np.array(img).astype(np.float32) / 255.0

    # Convert from HWC -> CHW (PyTorch format)
    arr = np.transpose(arr, (2, 0, 1))

    """apply ImageNet normalisation to match training preprocessing
    The model was trained with these exact values applied to the images,
    so we must apply them here too, or the pixel 
    values will be in the wrong range"""
    mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
    std  = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)
    arr  = (arr - mean) / std
    arr = arr.astype(np.float32)  # ensure float32 after normalisation, model doesn't accept float64

    
    # Add batch dimension → shape: (1, C, H, W)
    arr = np.expand_dims(arr, axis=0)

    return arr

print("✅ Image preprocessing function defined successfully")

In [ ]:
def predict(img_array):
    """
    Runs inference on a single preprocessed image.
    Add all your detection/segmentation/anomaly logic (if needed) here.
    Must return: (binary_prediction, probability/score)
    """

    # Run model inference using ONNX Runtime
    outputs = session.run(None, {input_name: img_array})
    logits = outputs[0]   # raw model output

    # ----- Default binary classification -----
    # Apply sigmoid to convert logits → probability
    prob = 1 / (1 + np.exp(-logits))[0][0]
    

    # Convert probability → binary decision
    pred = 1 if prob >= 0.55 else 0

    return pred, prob

print("✅ Prediction function defined successfully")

In [ ]:
# Gather all image paths (all .PNG images)
image_paths = sorted(
    glob.glob(os.path.join(IMAGE_DIR, "*.png"))
)

print(f"✅ Found {len(image_paths)} images for evaluation.")


# Start timer
start_time = time.time()

predictions = []

# Iterate over ALL images one-by-one
for path in image_paths:
    # Preprocessing (your implementation)
    img = preprocess_image(path)

    # Inference + postprocessing (your implementation)
    pred, prob = predict(img)

    # Store filename + prediction
    predictions.append((os.path.basename(path), pred))

# Stop timer
end_time = time.time()
total_runtime = end_time - start_time

# Display timing results
print("====================================")
print(f"✅ Inference runtime: {total_runtime:.4f} seconds")
print(f"Images processed:   {len(image_paths)}")
print(f"Avg per image:      {total_runtime / len(image_paths):.6f} sec")
print("====================================")

In [ ]:
# Convert predictions to DataFrame
df = pd.DataFrame(predictions, columns=["image_id", "target"])

# Save to CSV
df.to_csv("submission.csv", index=False)

print("✅ submission.csv created successfully!")
df.head()